In [1]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split


np.random.seed(42)
n_samples = 1000

data = pd.DataFrame({
    'ecg_hr': np.random.normal(75, 15, n_samples),
    'bp_sys': np.random.normal(120, 15, n_samples),
    'bp_dia': np.random.normal(80, 10, n_samples),
    'spo2': np.clip(np.random.normal(97, 3, n_samples), 70, 100),
    'temperature': np.random.normal(37.0, 0.8, n_samples),
    # Mocking RGB values (healthy is around [255, 234, 112])
    'urine_r': np.random.normal(255, 5, n_samples),
    'urine_g': np.random.normal(230, 20, n_samples),
    'urine_b': np.random.normal(110, 30, n_samples)
})


def assign_triage(row):
    # RED
    if row['spo2'] < 90 or row['ecg_hr'] > 130 or row['temperature'] > 39.5 or row['bp_sys'] > 180:
        return 2
    # YELLOW
    elif (row['spo2'] < 95) or (row['temperature'] > 38.0) or (row['bp_sys'] > 140):
        return 1
    # GREEN
    else:
        return 0

data['triage_class'] = data.apply(assign_triage, axis=1)

X = data.drop('triage_class', axis=1)
y = data['triage_class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# XGBoost Model
model = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=3,
    eval_metric='mlogloss',
    max_depth=3,
    learning_rate=0.1,
    n_estimators=50
)

model.fit(X_train, y_train)

# Save the model
model.save_model('triage_xgboost.json')
print("Model trained and saved as 'triage_xgboost.json'")

Model trained and saved as 'triage_xgboost.json'
